# Joins — Business Questions

Four analytics on silver tables. Each answer includes the query result, **join type**, and **why that join is correct**.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.joins.business_questions as bq_module
importlib.reload(bq_module)

from src.joins.business_questions import (
    SilverJoinTables,
    orders_without_customers,
    sellers_never_sold,
    top_delivered_orders_by_value,
    customers_in_main_and_late_arrivals,
    run_all_business_questions,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

tables = SilverJoinTables()
print("Loaded from:", bq_module.__file__)

## Q1 — Orders with no matching customer

**Join:** `left_anti` on `customer_id`

In [ ]:
q1_count, q1_join, q1_why = orders_without_customers(spark, tables)
print(f"Orders without customer: {q1_count}")
print(f"Join type: {q1_join}")
print(f"Why: {q1_why}")

## Q2 — Sellers who never sold anything

**Join:** `left_anti` on `seller_id`

In [ ]:
q2_count, q2_join, q2_why = sellers_never_sold(spark, tables)
print(f"Sellers never sold: {q2_count}")
print(f"Join type: {q2_join}")
print(f"Why: {q2_why}")

## Q3 — Top 10 delivered orders by value

**Join:** `inner` on `customer_id` and `order_id`

In [ ]:
q3_df, q3_join, q3_why = top_delivered_orders_by_value(spark, tables)
display(q3_df)
print(f"Join type: {q3_join}")
print(f"Why: {q3_why}")

## Q4 — Customers in main table and late-arrivals orders

**Join:** `inner` on `customer_id` (late-arrivals table has 99,441 orders — not empty)

In [ ]:
q4_df, q4_count, q4_join, q4_why = customers_in_main_and_late_arrivals(spark, tables)
print(f"Customers in both populations: {q4_count}")
display(q4_df.limit(10))
print(f"Join type: {q4_join}")
print(f"Why: {q4_why}")

In [ ]:
import json

report = run_all_business_questions(spark, tables)
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/joins_business_questions.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)